# 05: Querying the Graph

This notebook demonstrates how to query the Neo4j knowledge graph using graph patterns and Cypher queries.

## Prerequisites

**⚠️ Important:** Before running this notebook, ensure you have:
- Completed [**00-import.ipynb**](./00-import.ipynb) for environment detection and Neo4j connection setup
- Completed [**03-loading-data.ipynb**](./03-loading-data.ipynb) to load graph data into Neo4j

All environment detection, Neo4j connection, and configuration are handled in `00-import.ipynb`.

## Overview

This notebook shows you how to explore your knowledge graph by asking questions about the connections in your data. You can find what's related to what, discover patterns, and answer questions like "what projects are connected to this person?" or "show me all tasks related to this project."

We'll:
1. Query entities and relationships using graph patterns
2. Find related entities and explore the graph structure
3. Use graph patterns to answer questions


In [ ]:
# Run common imports and setup
%run 00-import.ipynb

# Additional imports specific to this notebook
from langchain.graphs import Neo4jGraph

print("✅ Additional libraries imported")


In [ ]:
# All setup is done in 00-import.ipynb
# Create Neo4jGraph for graph queries
kg = Neo4jGraph(
    url=settings.neo4j_uri,
    username=settings.neo4j_username,
    password=settings.neo4j_password,
    database=settings.neo4j_database
)

print(f"✅ Neo4jGraph initialized for graph queries")


## Query Entities

Find entities in the graph and explore their properties.


In [ ]:
# Query all entities
query = """
MATCH (e:Entity)
OPTIONAL MATCH (e)<-[:CONTAINS]-(n:Note)
RETURN e.name AS name, 
       e.type AS type,
       count(DISTINCT n) AS note_count
ORDER BY note_count DESC
LIMIT 20
"""

results = kg.query(query)
print("Top Entities:")
df = pd.DataFrame(results)
print(df)


In [ ]:
## Query Relationships

Find relationships between entities.


### Find Entity Relationships

Explore relationships between entities.


In [ ]:
# Query relationships
query = """
MATCH (e1:Entity)-[r]->(e2:Entity)
RETURN e1.name AS from_entity,
       type(r) AS relationship_type,
       e2.name AS to_entity,
       count(*) AS frequency
ORDER BY frequency DESC
LIMIT 20
"""

results = kg.query(query)
print("Entity Relationships:")
df = pd.DataFrame(results)
print(df)


## Find Related Entities

Find entities that are related to a specific entity.


In [ ]:
# Find entities related to a specific entity
entity_name = "Project"  # Change this to an entity from your graph

query = """
MATCH (e:Entity {name: $entity_name})-[r]-(related:Entity)
RETURN related.name AS related_entity,
       related.type AS entity_type,
       type(r) AS relationship_type,
       count(*) AS connection_count
ORDER BY connection_count DESC
LIMIT 10
"""

results = kg.query(query, params={"entity_name": entity_name})

if results:
    print(f"Entities related to '{entity_name}':")
    df = pd.DataFrame(results)
    print(df)
else:
    print(f"No related entities found for '{entity_name}'")
    print("Try a different entity name from your graph")


## Find Notes by Entity

Find notes that contain specific entities.


In [ ]:
# Find notes containing a specific entity
entity_name = "Project"  # Change this to an entity from your graph

query = """
MATCH (n:Note)-[:CONTAINS]->(e:Entity {name: $entity_name})
RETURN n.file_path AS file_path,
       n.file_name AS file_name,
       n.modified_at AS modified_at
ORDER BY n.modified_at DESC
LIMIT 10
"""

results = kg.query(query, params={"entity_name": entity_name})

if results:
    print(f"Notes containing '{entity_name}':")
    df = pd.DataFrame(results)
    print(df)
else:
    print(f"No notes found containing '{entity_name}'")


## Multi-Hop Queries

Find entities connected through multiple relationships.


In [ ]:
# Find entities connected through 2 hops
entity_name = "Project"  # Change this to an entity from your graph

query = """
MATCH (e:Entity {name: $entity_name})-[r1]-(intermediate:Entity)-[r2]-(target:Entity)
WHERE e <> intermediate <> target
RETURN target.name AS target_entity,
       target.type AS entity_type,
       type(r1) AS first_relationship,
       type(r2) AS second_relationship,
       intermediate.name AS intermediate_entity
LIMIT 10
"""

results = kg.query(query, params={"entity_name": entity_name})

if results:
    print(f"Entities 2 hops away from '{entity_name}':")
    df = pd.DataFrame(results)
    print(df)
else:
    print(f"No 2-hop connections found from '{entity_name}'")


## Next Steps

Proceed to:
- [**06-querying-vector-embeddings.ipynb**](./06-querying-vector-embeddings.ipynb): Query using vector similarity search
- [**07-knn-graphrag.ipynb**](./07-knn-graphrag.ipynb): Use KNN and graph data science for advanced graph-powered RAG
